<a href="https://colab.research.google.com/github/hum1-ua/Reto_Corref_PLN/blob/main/practica2_PLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reto 1: Extracción de Personajes - Procesamiento del Lenguaje Natural 1
**Grado en Inteligencia Artificial | Universidad de Alicante**

El objetivo de este reto es detectar y marcar en XML todas las menciones a personajes de la novela *Sab* de Gertrudis Gómez de Avellaneda, agrupándolas mediante un identificador único (correferencia) bajo el estándar TEI

In [ ]:
# 1. Instalación de librerías necesarias
# Se instala stanza para NLP y lxml para el parseo XML avanzado
!pip install torchao==0.18.0 # Dependencia necesaria para que funcione stanza
!pip install stanza lxml requests

import requests
from lxml import etree
import stanza

# 2. Descarga e inicialización del pipeline de Stanza para español
# Nota: La primera vez que se ejecuta, descargará los modelos (pesan un poco)
print("Descargando modelos de Stanza para español...")
stanza.download('es')

# Inicializamos el pipeline activando NER y correferencia (coref)
# tokenization y mwt (multi-word tokens) son necesarios como base
nlp = stanza.Pipeline(lang='es', processors='tokenize,mwt,pos,lemma,ner,coref')
print("Pipeline cargado correctamente.")

Descargando modelos de Stanza para español...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.15.0/resources/resources.json
INFO:stanza:Downloading default packages for language: es (Spanish) ...
INFO:stanza:File exists: /root/.cache/stanza/1.15.0/resources/es/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.15.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.15.0/resources/resources.json
INFO:stanza:Loading these models for language: es (Spanish):
| Processor | Package                  |
----------------------------------------
| tokenize  | combined_nocharlm        |
| mwt       | combined                 |
| pos       | combined_charlm          |
| lemma     | combined_nocharlm        |
| coref     | udcoref_xlm-roberta-lora |
| ner       | conll02                  |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: coref


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


Pipeline cargado correctamente.


In [ ]:
# URL obligatoria de la novela Sab
url_novela = "https://raw.githubusercontent.com/COST-ELTEC/ELTEC-spa/refs/heads/master/level1/SPA1021_GomezDeAvellaneda_Sab.xml"

# Descargamos el contenido
print(f"Descargando XML desde: {url_novela}")
respuesta = requests.get(url_novela)
respuesta.raise_for_status()

# Parseamos el XML recuperando el contenido binario para evitar problemas de codificación
parser = etree.XMLParser(remove_blank_text=False) # Vital mantener false para no alterar el formato
root = etree.fromstring(respuesta.content, parser)

# Los documentos TEI suelen usar un namespace por defecto. Debemos capturarlo para poder hacer búsquedas
namespace = {'tei': 'http://www.tei-c.org/ns/1.0'}

# Extraemos solo los nodos que representan capítulos, según la regla del atributo 'n'
# La expresión XPath busca los div de tipo chapter cuyo atributo n empiece por SPA1021
capitulos = root.xpath('//tei:div[@type="chapter" and starts-with(@n, "SPA1021")]', namespaces=namespace)

print(f"\nSe han detectado {len(capitulos)} capítulos en la novela.")

# Comprobación de que estamos leyendo bien el texto del primer capítulo
if capitulos:
    primer_capitulo = capitulos[0]
    num_capitulo = primer_capitulo.attrib.get('n')
    # Usamos itertext para sacar todo el texto plano sin etiquetas XML del capítulo
    texto_cap_1 = "".join(primer_capitulo.itertext())

    print(f"ID del primer capítulo extraído: {num_capitulo}")
    print(f"Muestra del texto crudo (primeros 500 caracteres):\n{texto_cap_1[:500]}...")

Descargando XML desde: https://raw.githubusercontent.com/COST-ELTEC/ELTEC-spa/refs/heads/master/level1/SPA1021_GomezDeAvellaneda_Sab.xml

Se han detectado 17 capítulos en la novela.
ID del primer capítulo extraído: SPA1021001
Muestra del texto crudo (primeros 500 caracteres):

					Capítulo I
					
					¿Quién eres? ¿cuál es tu patria?
					.....................
					.....................
					Las influencias tiranas
					de mi estrella, me formaron
					monstruo de especies tan raras,
					que gozo de heroica estirpe
					allá en las dotes del alma
					siendo el desprecio del mundo.
					CAÑIZARES
					
					Veinte años hace, poco más o menos, que al declinar una tarde
					del mes de junio un joven de hermosa presencia atravesaba a 
					caballo los campos pintore...


## 2. Segmentación en Memoria, Extracción y Filtrado (Mitigación OOM)

Los modelos de resolución de correferencia generan matrices de similitud entre todos los fragmentos (spans) posibles del texto. Esta complejidad espacial $O(N^2)$ provoca desbordamientos de RAM (Out Of Memory) si procesamos capítulos enteros en la capa gratuita de Colab[cite: 11].

Para viabilizar la ejecución, adoptamos una estrategia de **procesamiento por lotes basada en la estructura XML (Chunking por párrafos)**:
1. **Segmentación:** En lugar de extraer todo el texto del nodo `<div type="chapter">`[cite: 12], iteraremos sobre cada etiqueta interna `<p>` (párrafo).
2. **Procesamiento Acotado:** El pipeline de `Stanza` evaluará secuencias cortas, manteniendo el consumo de memoria estable. El *trade-off* de esta decisión es la pérdida de enlaces anafóricos inter-párrafo.
3. **Anclaje Espacial (NER) y Filtrado:** Aplicaremos la intersección de intervalos matemáticos. Si los *offsets* de la cadena de correferencia cruzan con los de una entidad `PER` del modelo NER, validamos la mención[cite: 8].
4. **Formateo TEI:** Asignaremos el ID alfanumérico secuencial exigido (`#pr00X`)[cite: 9] a cada clúster validado.

In [ ]:
# --- REGISTRO GLOBAL DE PERSONAJES ---
registro_personajes = {}
siguiente_id_global = 1

def procesar_parrafo_y_anotar_v3(nodo_parrafo, nlp_pipeline):
    global siguiente_id_global, registro_personajes
    texto = nodo_parrafo.text

    if not texto or len(texto.strip()) < 5:
        return

    doc = nlp_pipeline(texto)
    menciones_a_etiquetar = []

    # 1. EXTRACCIÓN NER DIRECTA (Nombres propios)
    entidades_per = [ent for ent in doc.ents if ent.type == 'PER']

    for ent in entidades_per:
        # Blindaje 1: Entidades malformadas por el tokenizador
        if ent.start_char is None or ent.end_char is None:
            continue

        nombre = ent.text.strip().lower()
        id_asignado = None

        for nombre_reg, id_reg in registro_personajes.items():
            if nombre in nombre_reg or nombre_reg in nombre:
                id_asignado = id_reg
                break

        if not id_asignado:
            id_asignado = f"#pr{siguiente_id_global:03d}"
            registro_personajes[nombre] = id_asignado
            siguiente_id_global += 1

        menciones_a_etiquetar.append({
            'start': ent.start_char,
            'end': ent.end_char,
            'id': id_asignado,
            'es_ner': True
        })

    # 2. EXTRACCIÓN DE CORREFERENCIA LOCAL (Pronombres y SN)[cite: 8]
    if hasattr(doc, 'coref') and doc.coref:
        for chain in doc.coref:
            id_cadena = None
            menciones_temp = []

            for mention in chain.mentions:
                oracion = doc.sentences[mention.sentence - 1]

                s_idx = mention.start_word[0] if isinstance(mention.start_word, tuple) else mention.start_word
                e_idx = mention.end_word[0] if isinstance(mention.end_word, tuple) else mention.end_word

                idx_inicio = max(0, min(s_idx - 1, len(oracion.words) - 1))
                idx_fin = max(0, min(e_idx - 2, len(oracion.words) - 1))

                i_char = oracion.words[idx_inicio].start_char
                f_char = oracion.words[idx_fin].end_char

                # Blindaje 2: Evitar el TypeError causado por los Multi-Word Tokens (MWT) de Stanza
                if i_char is None or f_char is None:
                    continue

                menciones_temp.append({'start': i_char, 'end': f_char})

                for m_ner in menciones_a_etiquetar:
                    if m_ner.get('es_ner'):
                        # Aseguramos matemáticamente la intersección solo sobre enteros
                        n_start, n_end = m_ner['start'], m_ner['end']
                        if max(i_char, n_start) < min(f_char, n_end):
                            id_cadena = m_ner['id']
                            break

            if id_cadena:
                for mt in menciones_temp:
                    if not any(m['start'] == mt['start'] and m['end'] == mt['end'] for m in menciones_a_etiquetar):
                        menciones_a_etiquetar.append({
                            'start': mt['start'],
                            'end': mt['end'],
                            'id': id_cadena
                        })

    # 3. FILTRADO GEOMÉTRICO E INYECCIÓN INVERSA
    if menciones_a_etiquetar:
        menciones_a_etiquetar = sorted(menciones_a_etiquetar, key=lambda x: x['start'], reverse=True)

        menciones_unicas = []
        for m in menciones_a_etiquetar:
            solape = False
            for mu in menciones_unicas:
                if max(m['start'], mu['start']) < min(m['end'], mu['end']):
                    solape = True
                    break
            if not solape:
                menciones_unicas.append(m)

        texto_modificado = texto
        for menc in menciones_unicas:
            inicio = menc['start']
            fin = menc['end']
            ref = menc['id']
            texto_modificado = (
                texto_modificado[:inicio] +
                f'<rs type="person" ref="{ref}">' +
                texto_modificado[inicio:fin] +
                '</rs>' +
                texto_modificado[fin:]
            )

        # 4. PARSEO DE NODOS (Corrección del problema &lt;rs&gt;)
        try:
            # Envolvemos en el namespace TEI original[cite: 9]
            fragmento_xml = f'<p xmlns="http://www.tei-c.org/ns/1.0">{texto_modificado}</p>'
            nuevo_nodo = etree.fromstring(fragmento_xml.encode('utf-8'))

            nodo_parrafo.clear()
            nodo_parrafo.text = nuevo_nodo.text
            for hijo in nuevo_nodo:
                nodo_parrafo.append(hijo)
        except Exception:
            pass # Si un carácter residual rompe el parseo XML en un párrafo aislado, se ignora sin detener el bucle

In [ ]:
import time

print("Iniciando procesamiento de la novela completa (esto tomará varios minutos)...")
inicio_tiempo = time.time()

# 1. Reiniciamos los contadores globales para la pasada final
registro_personajes.clear()
siguiente_id_global = 1

# 2. Iteración sobre toda la estructura de capítulos detectada en la ingesta[cite: 12]
for idx_cap, capitulo in enumerate(capitulos):
    print(f"Procesando Capítulo {idx_cap + 1} de {len(capitulos)}...")
    parrafos = capitulo.xpath('.//tei:p', namespaces=namespace)

    for p in parrafos:
        procesar_parrafo_y_anotar_v3(p, nlp)

# 3. Serialización y guardado del documento
# Empleamos tus apellidos formateados para el nombre del archivo de salida[cite: 12]
nombre_archivo = "SPA1021_GomezDeAvellaneda_Sab_AnotadoUrbanMartinez.xml"

# Reconstruimos el árbol completo (root) y lo guardamos con codificación UTF-8
arbol_modificado = etree.ElementTree(root)
arbol_modificado.write(nombre_archivo, encoding="utf-8", xml_declaration=True, pretty_print=True)

tiempo_total = time.time() - inicio_tiempo
print("-" * 50)
print(f"¡Procesamiento finalizado en {tiempo_total/60:.2f} minutos!")
print(f"Total de personajes/entidades únicos consolidados: {len(registro_personajes)}")
print(f"Archivo guardado exitosamente en Colab como: {nombre_archivo}")
print("Puedes descargarlo desde la pestaña de Archivos (icono de carpeta a la izquierda).")

Iniciando procesamiento de la novela completa (esto tomará varios minutos)...
Procesando Capítulo 1 de 17...
Procesando Capítulo 2 de 17...
Procesando Capítulo 3 de 17...
Procesando Capítulo 4 de 17...
Procesando Capítulo 5 de 17...
Procesando Capítulo 6 de 17...
Procesando Capítulo 7 de 17...
Procesando Capítulo 8 de 17...
Procesando Capítulo 9 de 17...
Procesando Capítulo 10 de 17...
Procesando Capítulo 11 de 17...
Procesando Capítulo 12 de 17...
Procesando Capítulo 13 de 17...
Procesando Capítulo 14 de 17...
Procesando Capítulo 15 de 17...
Procesando Capítulo 16 de 17...
Procesando Capítulo 17 de 17...
--------------------------------------------------
¡Procesamiento finalizado en 93.43 minutos!
Total de personajes/entidades únicos consolidados: 78
Archivo guardado exitosamente en Colab como: SPA1021_GomezDeAvellaneda_Sab_AnotadoUrbanMartinez.xml
Puedes descargarlo desde la pestaña de Archivos (icono de carpeta a la izquierda).
